# LAB 08 — Exercise 2 & Section 12: Tokenization and Subword Tokenization
**Môn học:** Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Giảng viên / TA:** Phạm Ngọc Hải  
**Sinh viên thực hiện:** Đỗ Khắc Đức Thành - MSSV: 24001697  

---
### Mục tiêu bài tập:
1. Hiểu và áp dụng **Tokenizer** của mô hình Pretrained Transformer (`bert-base-uncased`).
2. Phân tích chuỗi token, token IDs, special tokens và attention mask của câu: `"The cats are playing."`
3. So sánh số lượng từ (words), tokens và subword tokens.
4. Khám phá các khóa đầu ra của tokenizer (`input_ids`, `attention_mask`, `token_type_ids`) khi thiết lập `padding` và `truncation`.

## 1. Khởi tạo Tokenizer Pretrained (`bert-base-uncased`)

In [ ]:
from transformers import AutoTokenizer, BertTokenizer

# Tải Tokenizer chuẩn WordPiece của BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print(f"Special token IDs: {tokenizer.all_special_ids}")

## 2. Tokenize câu thực nghiệm: *"The cats are playing."*

In [ ]:
sentence = "The cats are playing."
print(f"Original Sentence: \"{sentence}\"")

# 1. Tách thành token thô (không có special tokens)
raw_tokens = tokenizer.tokenize(sentence)
print("\n1. Raw tokens (WordPiece):", raw_tokens)

# 2. Mã hóa đầy đủ (bao gồm special tokens và attention mask)
encoded = tokenizer(sentence)
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

# 3. Chuyển token IDs thành chuỗi token hoàn chỉnh
full_token_sequence = tokenizer.convert_ids_to_tokens(input_ids)

print("2. Full Token Sequence:", full_token_sequence)
print("3. Token IDs:", input_ids)
print("4. Attention Mask:", attention_mask)

### Chi tiết ánh xạ Token và ID:

In [ ]:
import pandas as pd

token_info = []
for idx, (tok, tid, mask) in enumerate(zip(full_token_sequence, input_ids, attention_mask)):
    role = "Special Token"
    if tok == "[CLS]":
        role = "[CLS] - Sequence classification / representation token"
    elif tok == "[SEP]":
        role = "[SEP] - Sequence separator / boundary token"
    elif tok == ".":
        role = "Punctuation"
    else:
        role = "Word / Subword Token"
        
    token_info.append({
        "Index": idx,
        "Token": tok,
        "Token ID": tid,
        "Attention Mask": mask,
        "Role": role
    })

df_tokens = pd.DataFrame(token_info)
df_tokens

## 3. So sánh Số Lượng (Quantity Comparison)
- **Words:** Số lượng từ trong câu tự nhiên.
- **Tokens:** Các token sau khi tách từ và dấu câu.
- **Subword Tokens:** Các đơn vị token thực tế được tokenizer mã hóa (tính cả special tokens).

In [ ]:
words = sentence.replace(".", "").split()
words_with_punct = sentence.split()

print(f"- Number of Words (từ ngữ thuần túy): {len(words)} -> {words}")
print(f"- Number of Words (tính cả dấu câu theo khoảng trắng): {len(words_with_punct)} -> {words_with_punct}")
print(f"- Number of Raw Tokens: {len(raw_tokens)} -> {raw_tokens}")
print(f"- Number of Subword Tokens (kèm [CLS], [SEP]): {len(full_token_sequence)} -> {full_token_sequence}")

## 4. Section 12: Core Implementation — Tokenization Pipeline
Khi chuẩn bị dữ liệu đầu vào cho Pretrained Transformer trong PyTorch, ta sử dụng:
```python
tokens = tokenizer(
    text,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
```

In [ ]:
sample_texts = [
    "The cat eats fish.",
    "BERT is a powerful bidirectional pretrained Transformer for natural language understanding."
]

batch_encoded = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Keys in batch_encoded:", batch_encoded.keys())
print("\n--- input_ids (Tensor shape: {batch_encoded['input_ids'].shape}) ---")
print(batch_encoded["input_ids"])

print("\n--- attention_mask (Tensor shape: {batch_encoded['attention_mask'].shape}) ---")
print(batch_encoded["attention_mask"])

if "token_type_ids" in batch_encoded:
    print("\n--- token_type_ids (Tensor shape: {batch_encoded['token_type_ids'].shape}) ---")
    print(batch_encoded["token_type_ids"])

### Giải thích các trường dữ liệu:
1. **`input_ids`:** Tensor chứa các chỉ số nguyên biểu diễn từng token trong bảng từ vựng (Vocabulary). Vị trí đệm (padding) nhận ID = 0 (`[PAD]`).
2. **`attention_mask`:** Tensor nhị phân (0 hoặc 1). Giá trị **1** đại diện cho token hợp lệ; giá trị **0** đại diện cho token padding `[PAD]`. Cơ chế Self-Attention sẽ áp dụng mặt nạ này để triệt tiêu trọng số chú ý tại các vị trí bằng 0, đảm bảo độ dài padding không làm sai lệch ý nghĩa ngữ cảnh.
3. **`token_type_ids`:** (Segment IDs) Dùng trong bài toán xử lý cặp câu (như Question-Answering, NLI). Các token thuộc câu A nhận nhãn 0, câu B nhận nhãn 1.

## 5. Khảo Sát Cơ Chế Tách Từ Subword (WordPiece)
Xem cách Tokenizer xử lý các từ phức tạp, từ hiếm hoặc từ mới xuất hiện ngoài từ điển thông thường:

In [ ]:
test_words = [
    "unbelievable",
    "Transformer",
    "tokenization",
    "antigravity",
    "catastrophic",
    "pretraining",
    "COVID19"
]

for w in test_words:
    subwords = tokenizer.tokenize(w)
    subword_ids = tokenizer.convert_tokens_to_ids(subwords)
    print(f"{w:<15} -> {subwords} (IDs: {subword_ids})")

**Nhận xét:**
- Ký hiệu `##` chỉ ra rằng token này là phần tiếp nối (suffix) của một từ, chứ không phải một từ độc lập bắt đầu bằng ký tự trắng.
- Ví dụ: `"unbelievable"` được tách thành `['un', '##belie', '##vable']`.
- Cơ chế này loại bỏ hoàn toàn hiện tượng token `[UNK]` (Unknown) đối với các từ phái sinh hoặc ghép từ, duy trì tính ổn định của mô hình ngôn ngữ.